# Teacher prompt

> The prompt that has Claude write the training message for each triage record

In [ ]:
#| default_exp teacher

The teacher is a large model, Claude, that writes the message for each synthetic triage record. Its messages are the training targets for the SLM, so this prompt defines what a good message is. It follows the structure of [write_prose](https://github.com/AnswerDotAI/aai-coding/blob/main/aai_coding/write_prose.py): rules, good examples, one bad message with its mistakes marked, a numbered list of failure modes, and plain words to use. The SLM never sees this prompt. Fine-tuning builds the behaviour into it, so the prompt's length only costs the teacher, and prompt caching covers most of that.

The failure modes are shared. The code checks in [`checks`](02_checks.ipynb) cover some of them, and the judges and the annotation app will use the same list for the rest, until expert error analysis replaces it.

In [ ]:
#| export
import json
from fastllm.chat import AsyncChat, contents

In [ ]:
#| hide
import re
from fastcore.test import *
from edgeslm.checks import *
from edgeslm.synth import *
from edgeslm.coffee import *

## Rules

The rules restate the message rules of the [scope](design/scope.ipynb) for the teacher. The triage contract keeps them simple: items arrive most severe first, and `since` is a phrase to copy.

In [ ]:
#| export
CONTEXT = """You write short WhatsApp messages to coffee farmers. Each message turns one triage record into actions. The record comes from monitoring code, which has already decided which problems matter, how severe they are and their likely cause. Only some plants carry a sensor, and each sensor stands for the plants around it."""

RULES = """- Write one paragraph per item, in the record's order.
- Start each paragraph with the item's marker (👀 early, ⚠️ moderate, 🚨 severe), then its location exactly as written in the record, then a colon.
- After the colon, say in plain words what is wrong and when it started, then what to do. Base the action on the likely cause, and state the cause as a fact when you mention it: the code has already weighed it. Never write likely, may, might or probably. When the cause is unknown, say so, then say what to check first.
- Say when the problem started with the record's `since` phrase, as it is.
- After the location, refer to it with words such as "this line", "this plot" or "its dripper". Never repeat the location or part of it.
- Write no digits outside the locations: no readings, counts or dates. Speak of monitored plants. Use `nodes_flagged` out of `nodes_in_group` to choose the amount: all, almost all (80% or more), most (more than half) or some (half or fewer). For a single node, say "a monitored plant" or "this plant".
- For water stress, say that plants are short of water or show first signs of water stress. For equipment faults, say what the sensors do wrong. Say that plants are not monitored only when their sensors send no data; readings with gaps still arrive, so never say it for them.
- For severe items, say that the action is for today.
- Be plain, friendly and direct. No greeting, no sign-off, no explanation of the monitoring.
- Keep the whole message under 600 characters."""

## Good examples

Six records from a training seed, each from a different tuple, with messages written by hand:

In [ ]:
#| export
EXAMPLES = [
    dict(record={"farm": "Sítio Bela Vista", "date": "2026-07-23", "nodes": 40, "items": [{"problem": "water stress", "location": "Block F, drip line F2", "pattern": "cluster", "severity": "early", "since": "since yesterday", "likely_cause": "low water pressure", "nodes_flagged": 3, "nodes_in_group": 6}, {"problem": "water stress", "location": "Block E, drip line E4", "pattern": "cluster", "severity": "early", "since": "since today", "likely_cause": "irrigation valve not opening", "nodes_flagged": 8, "nodes_in_group": 9}]},
         message='''👀 Block F, drip line F2: some monitored plants on this line have shown first signs of water stress since yesterday. Check the water pressure on the line.

👀 Block E, drip line E4: almost all monitored plants on this line have shown first signs of water stress since today, because its valve does not open. Check the valve when irrigation runs.'''),
    dict(record={"farm": "Fazenda Cachoeira", "date": "2026-05-07", "nodes": 30, "items": [{"problem": "water stress", "location": "Plot 1, row 2", "pattern": "one node", "severity": "severe", "since": "since Sunday", "likely_cause": "clogged dripper"}, {"problem": "water stress", "location": "Plot 2, row 8", "pattern": "one node", "severity": "moderate", "since": "since Tuesday", "likely_cause": "clogged dripper"}, {"problem": "equipment fault", "location": "Plot 4", "pattern": "cluster", "severity": "early", "since": "since today", "likely_cause": "readings from these nodes arrive intermittently", "nodes_flagged": 2, "nodes_in_group": 4}]},
         message='''🚨 Plot 1, row 2: a monitored plant has been short of water since Sunday and risks damage. Clear or replace its dripper today.

⚠️ Plot 2, row 8: a monitored plant has been short of water since Tuesday. Check its dripper for clogging.

👀 Plot 4: some sensors in this plot have sent readings with gaps since today. Check their batteries and antennas.'''),
    dict(record={"farm": "Fazenda Boa Vista", "date": "2026-05-09", "nodes": 53, "items": [{"problem": "equipment fault", "location": "whole farm", "pattern": "widespread", "severity": "severe", "since": "for a week or more", "likely_cause": "no data from most nodes: gateway radio problem", "nodes_flagged": 45, "nodes_in_group": 53}]},
         message='''🚨 Whole farm: almost all sensors have sent no data for a week or more, so the farm is not monitored. Check the gateway's power, antenna and radio today.'''),
    dict(record={"farm": "Sítio Santa Luzia", "date": "2026-05-26", "nodes": 19, "items": [{"problem": "equipment fault", "location": "whole farm", "pattern": "widespread", "severity": "early", "since": "since today", "likely_cause": "readings from most nodes arrive intermittently", "nodes_flagged": 14, "nodes_in_group": 19}]},
         message='''👀 Whole farm: most sensors have sent readings with gaps since today. Check that the gateway has power and a clear antenna.'''),
    dict(record={"farm": "Fazenda São João", "date": "2026-09-15", "nodes": 13, "items": [{"problem": "water stress", "location": "Bench 4, plant 1", "pattern": "one node", "severity": "early", "since": "since today", "likely_cause": "unknown"}, {"problem": "water stress", "location": "Bench 3, plant 3", "pattern": "one node", "severity": "early", "since": "since yesterday", "likely_cause": "clogged dripper"}]},
         message='''👀 Bench 4, plant 1: this plant has shown first signs of water stress since today, for no clear reason. Check that its dripper delivers water and that the soil around it is moist.

👀 Bench 3, plant 3: this plant has shown first signs of water stress since yesterday. Check its dripper for clogging.'''),
    dict(record={"farm": "Fazenda Cachoeira", "date": "2026-05-05", "nodes": 18, "items": [{"problem": "equipment fault", "location": "whole farm", "pattern": "widespread", "severity": "early", "since": "since yesterday", "likely_cause": "readings from most nodes arrive intermittently", "nodes_flagged": 13, "nodes_in_group": 18}, {"problem": "water stress", "location": "Plot 1, row 22", "pattern": "one node", "severity": "early", "since": "since yesterday", "likely_cause": "clogged dripper"}]},
         message='''👀 Whole farm: most sensors have sent readings with gaps since yesterday. Check that the gateway has power and a clear antenna.

👀 Plot 1, row 22: a monitored plant has shown first signs of water stress since yesterday. Check its dripper for clogging.''')]

Every example passes the code checks:

In [ ]:
test_eq([[k for k, v in check(e['record'], e['message']).items() if not v] for e in EXAMPLES], [[]]*len(EXAMPLES))

## A bad message

This message for the second example makes most of the mistakes the teacher must avoid. Each bracketed number refers to a failure mode in the next section, and the last line marks the item it leaves out:

In [ ]:
#| export
BAD = """Hello! [1][5] Our monitoring system has analysed your farm today. [2]

🚨 Plot 1, row 2: the node [3] shows severe [3] water stress since 3 May [4]. Plot 1, row 2 [8] may need attention. [9]

⚠️ [6] Plot 4: most [11] sensors in this cluster [3] might be sending data intermittently [3][9]. Please check your farm. [10]

Have a great day! [1]

[7: the item at Plot 2, row 8 is missing]"""

Without its markers, the bad message fails four of the code checks. The others, such as the greeting, the jargon and the hedging, need a judge:

In [ ]:
res = check(EXAMPLES[1]['record'], re.sub(r'\[[^\]]*\]', '', BAD))
test_eq({k for k, v in res.items() if not v}, {'format', 'coverage', 'severity', 'no_numbers'})
res

{'format': False,
 'coverage': False,
 'no_invented': True,
 'severity': False,
 'no_numbers': False,
 'length': True}

## Failure modes

Each failure mode has a short name, what goes wrong, and the fix. `checks` names the code checks that catch it, and is empty when a judge must decide.

In [ ]:
#| export
FAILURES = [
    dict(name='chat', checks=(), what='A greeting, a sign-off or small talk.', fix='Start with the first item and stop after the last.'),
    dict(name='system-talk', checks=(), what='Talk about the monitoring, the analysis or the data.', fix='Say what is wrong with the plants or the sensors.'),
    dict(name='jargon', checks=(), what='Words from the record or from engineering, such as node, cluster, severity, flagged or intermittently.', fix='Use the plain words listed below.'),
    dict(name='numbers', checks=('no_numbers',), what='Digits outside the locations: readings, counts, percentages or dates.', fix="Use the amount words and the record's `since` phrase."),
    dict(name='format', checks=('format',), what='A paragraph that does not start with a marker, the location and a colon, or two items in one paragraph.', fix='One paragraph per item, each starting with its marker and its location.'),
    dict(name='marker', checks=('severity',), what="A marker that does not match the item's severity.", fix='👀 early, ⚠️ moderate, 🚨 severe.'),
    dict(name='missing-or-invented', checks=('coverage', 'no_invented'), what='An item of the record left out, or a location not in the record.', fix='Write exactly one paragraph for each item.'),
    dict(name='repeated-location', checks=(), what='The location, or part of it, repeated after the colon.', fix='Refer to it as this line, this plot, its dripper.'),
    dict(name='hedging', checks=(), what='Likely, probably, may, might, possibly, it seems, about the problem or its cause.', fix='The code has decided. State the problem and its cause plainly.'),
    dict(name='vague-action', checks=(), what='No action, or one that does not follow from the likely cause.', fix='One concrete action that fits the cause, such as checking the line for a blockage.'),
    dict(name='wrong-amount', checks=(), what='An amount word that does not match the share of flagged sensors.', fix='All, almost all (80% or more), most (more than half), some (half or fewer).'),
    dict(name='wrong-tone', checks=(), what='Alarm for an early sign, or calm for a severe one.', fix='Match the urgency to the marker: say "today" for severe items.'),
    dict(name='too-long', checks=('length',), what='A message of more than 600 characters.', fix='One or two short sentences per item.')]

The bad message shows the first eleven. The last two need a longer message, or a calm one about a severe item.

Every code check belongs to a failure mode:

In [ ]:
test_eq({c for f in FAILURES for c in f['checks']}, set(check(EXAMPLES[0]['record'], EXAMPLES[0]['message'])))

## Plain words

The record's vocabulary leaks into messages unless the prompt names the plain word for each term:

In [ ]:
#| export
PLAIN = {'node': 'sensor, or the plant it monitors', 'cluster': 'the plants on this line, or in this plot or bench',
    'widespread': 'across the farm', 'flagged': 'affected, or showing signs', 'intermittently': 'with gaps',
    'implausible values': 'values that look wrong', 'severity, triage, pattern, likely cause': 'leave these words out'}

## The prompt

`system_prompt` assembles the parts in the order of write_prose: context, rules, good examples, the bad message, the failure modes, then plain words. `user_prompt` gives the record as JSON.

In [ ]:
#| export
def system_prompt()->str:
    "The teacher's system prompt"
    exs = '\n\n'.join(f"Record:\n{json.dumps(e['record'], ensure_ascii=False)}\n\nMessage:\n{e['message']}" for e in EXAMPLES)
    fails = '\n'.join(f"{i}. {f['what']} Fix: {f['fix']}" for i, f in enumerate(FAILURES, 1))
    plain = '\n'.join(f'- {k}: {v}' for k, v in PLAIN.items())
    return f"""{CONTEXT}

Rules:
{RULES}

Here are good messages for six records:

{exs}

Here is a bad message for the second record. Each bracketed number marks a mistake from the list that follows:

{BAD}

Mistakes to avoid:
{fails}

Plain words to use instead of the record's terms:
{plain}

Reply with the message only."""

def user_prompt(rec:dict)->str:
    "The teacher's prompt for one triage record"
    return json.dumps(rec, ensure_ascii=False)

In [ ]:
sp = system_prompt()
len(sp), sp[:300]

(8725,
 'You write short WhatsApp messages to coffee farmers. Each message turns one triage record into actions. The record comes from monitoring code, which has already decided which problems matter, how severe they are and their likely cause. Only some plants carry a sensor, and each sensor stands for the ')

## Writing messages

`teach` sends the prompt and one record to the teacher through [fastllm](https://github.com/AnswerDotAI/fastllm), which also reaches other providers and llama.cpp's OpenAI-compatible server, so the baseline and fine-tuned models can go through the same client later. Each call gets a fresh chat, so records never see each other. A temperature of 0.3 keeps the messages consistent and still varies their wording. Haiku is the default teacher, and `model` selects the second one, Sonnet, as the scope's Two teachers section describes.

In [ ]:
#| export
MODEL, TEMP = 'claude-haiku-4-5', 0.3

async def teach(
    rec:dict, # Triage record
    model:str=MODEL, # Teacher model
    temp:float=TEMP, # Sampling temperature
)->str: # The message for `rec`
    "Have the teacher write the message for `rec`"
    chat = AsyncChat(model, sp=system_prompt(), temp=temp, cache=True, cache_idxs=[0])
    r = await chat(user_prompt(rec), max_tokens=600)
    return ''.join(p.text for p in contents(r).content)

Calling the teacher needs `ANTHROPIC_API_KEY`, which `load_dotenv` reads from the project's untracked `.env` file. This cell does not run in tests:

In [ ]:
#| eval: false
from dotenv import load_dotenv
load_dotenv()
rec = sample(DIMS, triage, 10, seed=3)[7]['input']
await teach(rec)

For this record, with three moderate sensor faults, Haiku wrote:

> ⚠️ Plot 2, row 2: a sensor on this row has sent no data since Sunday because its battery is empty. Replace the battery today.
>
> ⚠️ Plot 3, row 7: a sensor on this row has sent no data since Saturday, so the plants it covers are not monitored. Check the sensor's power and antenna.
>
> ⚠️ Plot 4, row 23: a sensor on this row has sent no data since yesterday, so the plants it covers are not monitored. Check the sensor's power and antenna.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()